# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
contract_unit = """
Unit of analysis: one row = one content item (page), belonging to one
pseudonymized client. content_id is unique per row; client_id groups pages under
their client (32 distinct clients in this starter slice).

Time windows -- three, and they overlap on purpose:
  - Most fields (impressions_90d, ctr, avg_position, engagement_rate, etc.) are
    aggregated over a TRAILING 90-DAY WINDOW ending at export time.
  - clicks_last_30d / impressions_last_30d / sessions_last_30d cover the most
    RECENT 30 days of that same 90-day window.
  - clicks_prev_30d / impressions_prev_30d / sessions_prev_30d cover the 30 days
    BEFORE that (days 31-60 back) -- non-overlapping with last_30d.

This matters for ML-03's task: my label (declined = clicks_last_30d <
clicks_prev_30d) is defined on the last_30d vs prev_30d split. The 90-day
aggregates (ctr, avg_position, engagement_rate, ...) POOL ACROSS last_30d, so
they partially cover the same days my label is measuring -- that's a window
overlap I have to account for in the field classification below, not just note
and ignore.
"""
print(contract_unit)



Unit of analysis: one row = one content item (page), belonging to one
pseudonymized client. content_id is unique per row; client_id groups pages under
their client (32 distinct clients in this starter slice).

Time windows -- three, and they overlap on purpose:
  - Most fields (impressions_90d, ctr, avg_position, engagement_rate, etc.) are
    aggregated over a TRAILING 90-DAY WINDOW ending at export time.
  - clicks_last_30d / impressions_last_30d / sessions_last_30d cover the most
    RECENT 30 days of that same 90-day window.
  - clicks_prev_30d / impressions_prev_30d / sessions_prev_30d cover the 30 days
    BEFORE that (days 31-60 back) -- non-overlapping with last_30d.

This matters for ML-03's task: my label (declined = clicks_last_30d <
clicks_prev_30d) is defined on the last_30d vs prev_30d split. The 90-day
aggregates (ctr, avg_position, engagement_rate, ...) POOL ACROSS last_30d, so
they partially cover the same days my label is measuring -- that's a window
overlap I have t

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

field_contract = {
    # Feature -- knowable independent of / before the last_30d label window
    "content_type":            "Feature  -- static content metadata",
    "main_intent":              "Feature  -- static keyword-context metadata",
    "search_volume":            "Feature  -- static keyword-context metadata (blank = no keyword data)",
    "competition":               "Feature  -- static keyword-context metadata",
    "competition_level":         "Feature  -- static keyword-context metadata",
    "cpc":                        "Feature  -- static keyword-context metadata",
    "word_count":                 "Feature  -- content property, set when written",
    "char_count":                 "Feature  -- content property, set when written",
    "word_count_tier":            "Feature  -- derived from word_count",
    "char_count_tier":            "Feature  -- derived from char_count",
    "content_age_days":           "Feature  -- content property as of export",
    "age_tier":                   "Feature  -- derived from content_age_days",
    "age_tier_order":             "Feature  -- derived from content_age_days",
    "days_since_last_update":     "Feature  -- content property as of export",
    "freshness_tier":             "Feature  -- derived from days_since_last_update",
    "impressions_prev_30d":       "Feature  -- prior window, fully before the label window",
    "sessions_prev_30d":          "Feature  -- prior window, fully before the label window",

    # Label / proxy -- defines or falls inside the last_30d label window
    "clicks_last_30d":            "Label/proxy -- directly defines declined",
    "clicks_prev_30d":            "Label/proxy -- the other side of the declined comparison + eligibility filter",
    "impressions_last_30d":       "Label/proxy -- measured in the same window as the label",
    "sessions_last_30d":          "Label/proxy -- measured in the same window as the label",
    "trend_direction":            "Label/proxy -- the pipeline's own decline label, never a feature",
    "trend_pct":                  "Label/proxy -- computed from last_30d vs prev_30d impressions, never a feature",

    # Context -- for grouping/joining/splitting only
    "content_id":                 "Context -- pseudonymous id, grouping/joins only",
    "client_id":                  "Context -- pseudonymous id, use for CLIENT-HOLDOUT splits only",

    # Excluded -- window overlap or explicitly flagged
    "ctr":                        "Excluded -- 90d window overlaps the last_30d label window (leaks label period)",
    "avg_position":               "Excluded -- 90d window overlaps the last_30d label window (leaks label period)",
    "position_tier":              "Excluded -- derived from avg_position, same overlap",
    "engagement_rate":            "Excluded -- 90d window overlaps the last_30d label window",
    "scroll_rate":                "Excluded -- 90d window overlaps the last_30d label window",
    "ai_traffic_pct":             "Excluded -- 90d window overlaps the last_30d label window",
    "impressions_90d":            "Excluded -- 90d window overlaps the last_30d label window",
    "clicks_90d":                 "Excluded -- 90d window overlaps the last_30d label window",
    "pageviews_90d":              "Excluded -- 90d window overlaps the last_30d label window",
    "sessions_90d":                "Excluded -- 90d window overlaps the last_30d label window",
    "provider_used":              "Excluded -- data dictionary flags this explicitly: not a model feature",
    "model_used":                 "Excluded -- data dictionary flags this explicitly: not a model feature",
}

contract_df = pd.DataFrame(
    [(col, note.split(" -- ")[0].split("/")[0].strip(), note) for col, note in field_contract.items()],
    columns=["column", "bucket", "why"],
)
print(contract_df["bucket"].value_counts())
contract_df


bucket
Feature     17
Excluded    12
Label        6
Context      2
Name: count, dtype: int64


,column,bucket,why
0,content_type,Feature,Feature -- static content metadata
1,main_intent,Feature,Feature -- static keyword-context metadata
2,search_volume,Feature,Feature -- static keyword-context metadata (b...
3,competition,Feature,Feature -- static keyword-context metadata
4,competition_level,Feature,Feature -- static keyword-context metadata
5,cpc,Feature,Feature -- static keyword-context metadata
6,word_count,Feature,"Feature -- content property, set when written"
7,char_count,Feature,"Feature -- content property, set when written"
8,word_count_tier,Feature,Feature -- derived from word_count
9,char_count_tier,Feature,Feature -- derived from char_count


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
print("=== GRAIN ===")
print("total rows:", len(df))
print("unique content_id:", df["content_id"].nunique())
dup_ids = df["content_id"].value_counts()
print("content_ids appearing more than once:", (dup_ids > 1).sum(), "-> grain holds")
print("unique client_id:", df["client_id"].nunique(), "(claimed: 32)")

print()
print("=== COUNTS PER CLIENT (unbalanced, as expected) ===")
print(df.groupby("client_id").size().describe())

print()
print("=== MISSINGNESS -- overall ===")
print("search_volume nulls:", df["search_volume"].isna().sum(), "(dictionary claims 2,468)")
print("word_count nulls:", df["word_count"].isna().sum(), "(dictionary claims 7,699)")
print("trend_pct nulls:", df["trend_pct"].isna().sum(), "(dictionary claims 3,388)")
print("avg_position == 0 count:", (df["avg_position"] == 0).sum(), "(dictionary claims 1,205 -- means NO position data)")

print()
print("=== MISSINGNESS -- patterned, not random (by content_type) ===")
print((df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean() * 100)).round(1).astype(str) + "%")

print()
print("=== WINDOW SANITY ===")
print("content_age_days min:", df["content_age_days"].min(), "(dictionary claims every row >= 90)")
print("age_tier values present:", sorted(df["age_tier"].unique()))
print("impressions_90d min:", df["impressions_90d"].min(), "(dictionary claims every row >= 1)")


=== GRAIN ===
total rows: 30000
unique content_id: 30000
content_ids appearing more than once: 0 -> grain holds
unique client_id: 32 (claimed: 32)

=== COUNTS PER CLIENT (unbalanced, as expected) ===
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64

=== MISSINGNESS -- overall ===
search_volume nulls: 2468 (dictionary claims 2,468)
word_count nulls: 7699 (dictionary claims 7,699)
trend_pct nulls: 3388 (dictionary claims 3,388)
avg_position == 0 count: 1205 (dictionary claims 1,205 -- means NO position data)

=== MISSINGNESS -- patterned, not random (by content_type) ===
content_type
comparison article      0.0%
feedly article        100.0%
keyword article         1.4%
Name: search_volume, dtype: str

=== WINDOW SANITY ===
content_age_days min: 90 (dictionary claims every row >= 90)
age_tier values present: ['181-365', '31-90', '365+', '91-180']
impression

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
data_limits = """
What this starter slice can NEVER tell me:

1. No real calendar dates. Every field is a pre-aggregated window (90d / last_30d
   / prev_30d) with no date column -- I can verify window LENGTHS but not verify
   that all 32 clients' windows line up on the same calendar days. The warehouse
   docs warn client histories are an "unbalanced panel" (some clients have 17
   months, some 3) -- this CSV can't confirm or rule that out; I'd need
   dim_clients.gsc_data_start from the full warehouse for that.

2. Missingness is patterned, not random. search_volume is 100% missing for
   'feedly article' rows and ~0% missing for 'comparison article' rows (verified
   above) -- a blind fillna(0) would silently teach a model "feedly article" via
   the missingness pattern instead of the actual column.

3. avg_position == 0 means "no ranking data", not literal position zero (1,205
   rows, verified above) -- and I've excluded avg_position/position_tier entirely
   anyway, due to the last_30d window-overlap issue in section 2.

4. Association only, never causation. Nothing here was randomized -- I can
   observe that a signal moves with decline, never that changing it would cause
   a page to stop declining.

5. provider_used / model_used are excluded per the dictionary's own note ("not a
   model feature") -- I'm treating that as ops/production metadata that could
   proxy for which team or era produced the content, not a genuine SEO signal.

6. This 30k-row CSV is a teaching slice of a ~79M-row warehouse -- any pattern
   found here is a hypothesis to re-check against the full warehouse later, not
   a client-wide conclusion yet.
"""
print(data_limits)



What this starter slice can NEVER tell me:

1. No real calendar dates. Every field is a pre-aggregated window (90d / last_30d
   / prev_30d) with no date column -- I can verify window LENGTHS but not verify
   that all 32 clients' windows line up on the same calendar days. The warehouse
   docs warn client histories are an "unbalanced panel" (some clients have 17
   months, some 3) -- this CSV can't confirm or rule that out; I'd need
   dim_clients.gsc_data_start from the full warehouse for that.

2. Missingness is patterned, not random. search_volume is 100% missing for
   'feedly article' rows and ~0% missing for 'comparison article' rows (verified
   above) -- a blind fillna(0) would silently teach a model "feedly article" via
   the missingness pattern instead of the actual column.

3. avg_position == 0 means "no ranking data", not literal position zero (1,205
   rows, verified above) -- and I've excluded avg_position/position_tier entirely
   anyway, due to the last_30d window-ov

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.